# Gene mutation feature extraction

Due to the discrete nature of gene mutation data, the aforementioned method could not be applied. Consequently, we adopted a Chi-square-G test approach, which is consistent with our strategy for processing gene expression data, namely, exploring the differences in mutations between genes within and outside specific pathways. The detailed processing steps for gene mutation data are as follows: First, we tallied the occurrences of damaging mutations in pathway gene sets within and outside cell lines, constructing a contingency table as shown in Table 4. Subsequently, we employed the Chi-square-G test method to calculate the deviation between the observed and expected frequencies based on this contingency table, obtaining the chi-square value. The chi-square value represents the magnitude of the difference in gene mutations between genes within and outside the gene set in cell lines. A larger chi-square value implies a greater difference in gene mutation occurrences between within and outside of the gene set. Ultimately, we utilized the chi-square statistic obtained through this process as the gene mutation feature for cell lines.


![Table 4. Gene mutation contingency table](img/img.png)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats.contingency import expected_freq
from scipy.stats import power_divergence
# Degrees of freedom.
dof = 1
count1 = 0
count2 = 0
count3 = 0
count_zero = 0

In [3]:
# df_gene_set = pd.read_csv('data/GOgeneSet.tsv', sep='\t',header=None)
df_gene_set = pd.read_csv('data/c2.cp.kegg_legacy.v2023.2.Hs.symbols.csv',header=None)
df_mutation = pd.read_csv('data/CCLE_mutations_bool_damaging_modified.csv')
print(df_mutation.shape)
print(df_gene_set.shape)

(1758, 17347)
(186, 390)


In [4]:
df_mutation = pd.read_csv('data/CCLE_mutations_bool_damaging_modified.csv',index_col=0)
print(df_mutation.shape)
# df_mutation.head(5)

(1758, 17346)


In [5]:

# Remove cell lines where no mutations occurred in all genes.
df_mutation_temp  = df_mutation.iloc[:, 1:]
df_mutation_temp_sum = df_mutation_temp.sum(axis=1)
# Check if the sum is greater than 0.
mut_bool = df_mutation_temp_sum.gt(0)
# Filter out cell lines with gene mutations.
df_mutation = df_mutation.loc[mut_bool]
print(df_mutation.shape)
print(df_gene_set.shape)

(1757, 17346)
(186, 390)


In [6]:

# all = df_mutation.iloc[:,1:].apply(lambda x: x.sum(), axis=1)
# Iterate through the gene set and calculate the chi-square value for each cell line.
genes = df_mutation.columns[1:]

gene_sets = df_gene_set

mutation_genes = set(df_mutation.iloc[:,1:].columns.values)
# Retrieve the name of the cell line.
cell_line_names = df_mutation.iloc[:,0]

In [7]:
# Chi-square-G Test
def Chi_square_G_Test(r):
    expected_matrix = expected_freq(r)
    # Check if there are any zeros in the expected_matrix.
    if np.any(expected_matrix == 0):
        # Set the chi-square value directly to 0.
        # print('expected_matrix exist 0')
        return 0
        
    # Check if there are any numbers less than 1 in the matrix.
    if np.any(expected_matrix < 5):
        # Use the G-test for maximum likelihood statistical significance testing, which is approximately equivalent to the chi-square test.
        x2 = power_divergence(r,expected_matrix, ddof=r.size - 2, axis=None,lambda_='log-likelihood')[0]
        return x2
    else:
        # No continuity correction is needed.
        x2 = np.sum((r - expected_matrix) ** 2 / expected_matrix)
    return x2

In [8]:
count_zero = 0
#Store the chi-square value.
Cardinality_analysis_of_variance = []
for i,data in enumerate(gene_sets.values):

    gene_set_name = data[0]

    gene_set = set(data[1:]) 

    if np.nan in gene_set:
        gene_set.remove(np.nan)

    if not mutation_genes & gene_set: 
        # If there is no intersection between `mutation_genes` and `gene_set`, skip this iteration and proceed to the next loop.
        gene_sets.drop(index=[i],inplace=True)
        print('drop',i,gene_set_name)
        continue
        
    differences = mutation_genes - gene_set 

    intersections = mutation_genes & gene_set
    # Internal data of the gene set.
    internal = df_mutation[list(intersections)]
    # External data of the gene set.
    external = df_mutation[list(differences)]

    internal_row_sum = internal.shape[1]
    external_row_sum = external.shape[1]

    temp_lst = [internal.shape[1] for _ in range(internal.shape[0])]
    internal_temp_lst = pd.Series(temp_lst)

    internal_temp_lst.reset_index(drop=True,inplace=True)

    temp_lst = [external.shape[1] for _ in range(external.shape[0])]
    external_temp_lst = pd.Series(temp_lst)
    external_temp_lst.reset_index(drop=True,inplace=True)

    # Number of mutations inside and outside the gene set.
    internal_mutation_sum = internal.sum(axis=1)
    
    internal_mutation_sum.reset_index(drop=True,inplace=True)
    external_mutation_sum = external.sum(axis=1) 
    
    external_mutation_sum.reset_index(drop=True,inplace=True)

    # Number of non-mutations inside and outside the gene set.
    internal_not_mutation_sum = internal_temp_lst - internal_mutation_sum
    external_not_mutation_sum = external_temp_lst - external_mutation_sum

    # Iterate through each cell line, calculate the chi-square value, and record it in the container.
    X_square = []
    for j in range(df_mutation.shape[0]): # Here is the number of cell lines.
        R = np.array([[internal_mutation_sum.iloc[j],internal_not_mutation_sum.iloc[j]],
                      [external_mutation_sum.iloc[j],external_not_mutation_sum.iloc[j]]])

        x2 = Chi_square_G_Test(R)
        if x2 == 0 :
#             print(R)
            count_zero = count_zero + 1
            print(count_zero)
        X_square.append(x2)

    # print(X_square)
    print(i+1,gene_set_name)
    Cardinality_analysis_of_variance.append(X_square)
print('Done!')

1 KEGG_ABC_TRANSPORTERS
2 KEGG_ACUTE_MYELOID_LEUKEMIA
3 KEGG_ADHERENS_JUNCTION
4 KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY
5 KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM
6 KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION
7 KEGG_ALLOGRAFT_REJECTION
8 KEGG_ALPHA_LINOLENIC_ACID_METABOLISM
9 KEGG_ALZHEIMERS_DISEASE
10 KEGG_AMINOACYL_TRNA_BIOSYNTHESIS
11 KEGG_AMINO_SUGAR_AND_NUCLEOTIDE_SUGAR_METABOLISM
12 KEGG_AMYOTROPHIC_LATERAL_SCLEROSIS_ALS
13 KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION
14 KEGG_APOPTOSIS
15 KEGG_ARACHIDONIC_ACID_METABOLISM
16 KEGG_ARGININE_AND_PROLINE_METABOLISM
17 KEGG_ARRHYTHMOGENIC_RIGHT_VENTRICULAR_CARDIOMYOPATHY_ARVC
18 KEGG_ASCORBATE_AND_ALDARATE_METABOLISM
19 KEGG_ASTHMA
20 KEGG_AUTOIMMUNE_THYROID_DISEASE
21 KEGG_AXON_GUIDANCE
22 KEGG_BASAL_CELL_CARCINOMA
23 KEGG_BASAL_TRANSCRIPTION_FACTORS
24 KEGG_BASE_EXCISION_REPAIR
25 KEGG_BETA_ALANINE_METABOLISM
26 KEGG_BIOSYNTHESIS_OF_UNSATURATED_FATTY_ACIDS
27 KEGG_BLADDER_CANCER
28 KEGG_BUTANOATE_METABOLISM
29 KEGG_B_CELL_RECEPTO

In [9]:
# row: pathway ID; column: cell line name sample.
row_GO = gene_sets.iloc[:, 0]
col_cell_line_names = cell_line_names.index
df_Cardinality_analysis_of_variance = pd.DataFrame(Cardinality_analysis_of_variance,index=row_GO,
                                                   columns=col_cell_line_names).T

In [10]:
df_Cardinality_analysis_of_variance.to_csv('MUT_cardinality_analysis_of_variance_KEGG186.csv',index=True)